# Taxi Trip Fetching

In this notebook we fetch the taxi trip data provided by the city of chicago: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data (downloaded on 05.05.2026).

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import numpy as np

from sodapy import Socrata
import requests

import time

In [3]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch data info                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

URL = "https://data.cityofchicago.org/resource/ajtu-isnz.json"
params = {
    "$select": "count(*) AS n",
    "$where": "trip_start_timestamp >= '2025-01-01T00:00:00' "
              "AND trip_start_timestamp < '2026-01-01T00:00:00'",
}
print(requests.get(URL, params=params, timeout=120).json())

[{'n': '6825838'}]


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch data                              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import io

API_ENDPOINT = "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
LIMIT = 50000
MAX_RETRIES = 5
OUTPUT_FILE = "CHICAGO_TAXI_TRIPS_2025.csv"

def fetch_page(last_ts, last_id):
    # Keyset pagination: jump directly to where we left off using an index-friendly filter.
    # Avoids the O(n) row-skip cost of offset-based pagination.
    if last_ts is None:
        where = (
            "trip_start_timestamp >= '2025-01-01T00:00:00' "
            "AND trip_start_timestamp < '2026-01-01T00:00:00'"
        )
    else:
        where = (
            f"trip_start_timestamp < '2026-01-01T00:00:00' "
            f"AND (trip_start_timestamp > '{last_ts}' "
            f"OR (trip_start_timestamp = '{last_ts}' AND trip_id > '{last_id}'))"
        )
    params = {
        "$where": where,
        "$order": "trip_start_timestamp, trip_id",
        "$limit": LIMIT,
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(API_ENDPOINT, params=params, timeout=120, stream=True) as r:
                r.raise_for_status()
                return r.content
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            print(f"  Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

last_ts, last_id = None, None
first_page = True
total_rows = 0

with open(OUTPUT_FILE, "wb") as f:
    while True:
        print(f"Fetching next page (cursor: {last_ts})...")
        content = fetch_page(last_ts, last_id)

        df = pd.read_csv(io.BytesIO(content))
        row_count = len(df)
        print(f"  Got {row_count} rows. Total so far: {total_rows + row_count}")

        if row_count == 0:
            break

        if first_page:
            df.to_csv(f, index=False)
            first_page = False
        else:
            df.to_csv(f, index=False, header=False)

        total_rows += row_count
        last_ts = df["trip_start_timestamp"].iloc[-1]
        last_id = df["trip_id"].iloc[-1]

        if row_count < LIMIT:
            break

print(f"Download complete. {total_rows} rows written to {OUTPUT_FILE}.")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Resume fetch from last saved record     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import io

API_ENDPOINT = "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
LIMIT = 50000
MAX_RETRIES = 5
OUTPUT_FILE = "CHICAGO_TAXI_TRIPS_2025.csv"

# Read only the last row of the existing CSV to get the cursor
existing = pd.read_csv(OUTPUT_FILE)
total_rows = len(existing)
last_ts = existing["trip_start_timestamp"].iloc[-1]
last_id = existing["trip_id"].iloc[-1]
print(f"Resuming after {total_rows} rows. Last cursor: {last_ts}, trip_id={last_id}")
del existing  # free memory

def fetch_page(last_ts, last_id):
    where = (
        f"trip_start_timestamp < '2026-01-01T00:00:00' "
        f"AND (trip_start_timestamp > '{last_ts}' "
        f"OR (trip_start_timestamp = '{last_ts}' AND trip_id > '{last_id}'))"
    )
    params = {"$where": where, "$order": "trip_start_timestamp, trip_id", "$limit": LIMIT}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(API_ENDPOINT, params=params, timeout=120, stream=True) as r:
                r.raise_for_status()
                return r.content
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            print(f"  Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

with open(OUTPUT_FILE, "ab") as f:
    while True:
        print(f"Fetching next page (cursor: {last_ts})...")
        content = fetch_page(last_ts, last_id)

        df = pd.read_csv(io.BytesIO(content))
        row_count = len(df)
        print(f"  Got {row_count} rows. Total so far: {total_rows + row_count}")

        if row_count == 0:
            break

        df.to_csv(f, index=False, header=False)
        total_rows += row_count
        last_ts = df["trip_start_timestamp"].iloc[-1]
        last_id = df["trip_id"].iloc[-1]

        if row_count < LIMIT:
            break

print(f"Resume complete. {total_rows} rows total in {OUTPUT_FILE}.")